# LungNet22 Model - Versi Baru (6 Classes)
Skenario B: Balanced Data (dengan GAN + Augmentasi) - Disesuaikan dengan Panduan Project ILMED

Dataset: `chest-xray-preprocessed` (Kaggle Chest X-Ray 6 Classes yang telah di-preprocess offline dengan CLAHE)
- 6 Kelas: Covid-19, Emphysema, Normal, Pneumonia-Bacterial, Pneumonia-Viral, Tuberculosis
- Preprocessing Offline: Image Enhancement (CLAHE clipLimit=3, tileGridSize=(10,10)) telah diterapkan offline untuk menghindari bottleneck CPU saat training.
- Dataset: Penggabungan data Val & Test (sesuai instruksi Asdos)
- Balancing: **Conditional GAN (cGAN)** digunakan untuk men-generate gambar sintetis kelas minoritas agar seimbang dengan kelas mayoritas (`Normal` = 2.671).
- Backbone Classifier: VGG16 (Pretrained ImageNet)
- Penambahan: Block 6 & Block 7 (Dense 6)
- Evaluasi Lengkap: Akurasi/Loss, Confusion Matrix, Metrik per kelas (termasuk Specificity), ROC/AUC, Grad-CAM & Score-CAM


### 1. Impor Library & Cek Environment


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import glob
import cv2
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from sklearn.preprocessing import label_binarize

print("TensorFlow Version:", tf.__version__)
# Cek ketersediaan GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("GPU is available:")
    for gpu in gpus:
        print("  -", gpu)
    # Aktifkan Mixed Precision untuk akselerasi training GPU
    from tensorflow.keras import mixed_precision
    policy = mixed_precision.Policy('mixed_float16')
    mixed_precision.set_global_policy(policy)
    print("Mixed precision (float16) enabled.")
else:
    print("GPU is NOT available. Running on CPU.")


### 2. Memuat Dataset Paths (Preprocessed Folder) & Menggabungkan Data Val + Test (Instruksi Asdos)


In [ ]:
base_dir = './chest-xray-preprocessed' # Menggunakan dataset yang sudah di-CLAHE secara offline
train_dir = os.path.join(base_dir, 'train')
val_dir = os.path.join(base_dir, 'val')
test_dir = os.path.join(base_dir, 'test')

# Fungsi pembantu untuk membuat dataframe berisi path file dan label
def get_data_df(directory):
    filepaths = []
    labels = []
    classes = os.listdir(directory)
    for cl in classes:
        cl_path = os.path.join(directory, cl)
        if os.path.isdir(cl_path):
            files = glob.glob(os.path.join(cl_path, "*"))
            for f in files:
                if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                    filepaths.append(f)
                    labels.append(cl)
    return pd.DataFrame({'Filepath': filepaths, 'Label': labels})

# Muat dataframe
train_df = get_data_df(train_dir)
val_df = get_data_df(val_dir)
test_df = get_data_df(test_dir)

# Gabungkan Val & Test sesuai instruksi Asdos
val_test_df = pd.concat([val_df, test_df], axis=0).reset_index(drop=True)

# Hitung jumlah kelas untuk pemetaan label
class_names = sorted(list(train_df['Label'].unique()))
class_to_idx = {name: i for i, name in enumerate(class_names)}
idx_to_class = {i: name for i, name in enumerate(class_names)}

print("Pemetaan Kelas:", class_to_idx)
print(f"Jumlah data Train                  : {len(train_df)} file")
print(f"Jumlah data Val & Test (Gabungan)  : {len(val_test_df)} file")


### 3. Membangun Arsitektur Conditional GAN (cGAN)


In [ ]:
# Model Generator cGAN
def build_generator(latent_dim, num_classes):
    noise_input = tf.keras.layers.Input(shape=(latent_dim,), name='noise_input')
    label_input = tf.keras.layers.Input(shape=(1,), name='label_input')
    
    # Label Embedding
    label_embed = tf.keras.layers.Embedding(num_classes, 50)(label_input)
    label_embed = tf.keras.layers.Flatten()(label_embed)
    label_embed = tf.keras.layers.Dense(7 * 7, activation='relu')(label_embed)
    label_embed = tf.keras.layers.Reshape((7, 7, 1))(label_embed)
    
    # Noise project to 7x7x512
    noise_dense = tf.keras.layers.Dense(7 * 7 * 512, activation='relu')(noise_input)
    noise_dense = tf.keras.layers.Reshape((7, 7, 512))(noise_dense)
    
    # Concatenate
    concat = tf.keras.layers.Concatenate(axis=-1)([noise_dense, label_embed])
    
    # Upsample to 14x14
    x = tf.keras.layers.Conv2DTranspose(256, (4, 4), strides=(2, 2), padding='same', use_bias=False)(concat)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Upsample to 28x28
    x = tf.keras.layers.Conv2DTranspose(128, (4, 4), strides=(2, 2), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Upsample to 56x56
    x = tf.keras.layers.Conv2DTranspose(64, (4, 4), strides=(2, 2), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Upsample to 112x112
    x = tf.keras.layers.Conv2DTranspose(32, (4, 4), strides=(2, 2), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Upsample to 224x224 (dtype='float32' untuk mixed precision stability)
    out_img = tf.keras.layers.Conv2DTranspose(1, (4, 4), strides=(2, 2), padding='same', activation='tanh', dtype='float32', name='out_img')(x)
    
    return tf.keras.Model(inputs=[noise_input, label_input], outputs=out_img, name='generator')

# Model Discriminator cGAN
def build_discriminator(img_shape, num_classes):
    img_input = tf.keras.layers.Input(shape=img_shape, name='img_input')
    label_input = tf.keras.layers.Input(shape=(1,), name='label_input')
    
    # Label Embedding
    label_embed = tf.keras.layers.Embedding(num_classes, 50)(label_input)
    label_embed = tf.keras.layers.Flatten()(label_embed)
    label_embed = tf.keras.layers.Dense(img_shape[0] * img_shape[1], activation='relu')(label_embed)
    label_embed = tf.keras.layers.Reshape((img_shape[0], img_shape[1], 1))(label_embed)
    
    # Concatenate image and label channel
    concat = tf.keras.layers.Concatenate(axis=-1)([img_input, label_embed])
    
    # Downsample to 112x112
    x = tf.keras.layers.Conv2D(64, (4, 4), strides=(2, 2), padding='same')(concat)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Downsample to 56x56
    x = tf.keras.layers.Conv2D(128, (4, 4), strides=(2, 2), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Downsample to 28x28
    x = tf.keras.layers.Conv2D(256, (4, 4), strides=(2, 2), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Downsample to 14x14
    x = tf.keras.layers.Conv2D(512, (4, 4), strides=(2, 2), padding='same', use_bias=False)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(0.2)(x)
    
    # Classifier out (dtype='float32' untuk mixed precision stability)
    x = tf.keras.layers.Flatten()(x)
    out_validity = tf.keras.layers.Dense(1, dtype='float32', name='out_validity')(x)
    
    return tf.keras.Model(inputs=[img_input, label_input], outputs=out_validity, name='discriminator')

LATENT_DIM = 100
IMG_SHAPE = (224, 224, 1)
NUM_CLASSES = 6

generator = build_generator(LATENT_DIM, NUM_CLASSES)
discriminator = build_discriminator(IMG_SHAPE, NUM_CLASSES)

generator.summary()
discriminator.summary()


### 4. Custom Training Loop cGAN


In [ ]:
# Optimizer & Loss
g_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
d_optimizer = tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5)
cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)

def generator_loss(fake_output):
    return cross_entropy(tf.ones_like(fake_output), fake_output)

def discriminator_loss(real_output, fake_output):
    real_loss = cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss

@tf.function
def train_step(images, labels):
    batch_size = tf.shape(images)[0]
    noise = tf.random.normal([batch_size, LATENT_DIM])
    
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator([noise, labels], training=True)
        real_output = discriminator([images, labels], training=True)
        fake_output = discriminator([generated_images, labels], training=True)
        
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)
        
    gradients_of_generator = gen_tape.gradient(gen_loss, generator.trainable_variables)
    gradients_of_discriminator = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    
    g_optimizer.apply_gradients(zip(gradients_of_generator, generator.trainable_variables))
    d_optimizer.apply_gradients(zip(gradients_of_discriminator, discriminator.trainable_variables))
    
    return gen_loss, disc_loss


### 5. Memuat Gambar Latihan ke Memori (Sangat cepat karena sudah ter-CLAHE offline)


In [ ]:
print("Memuat dataset training ke memori untuk training cGAN...")
X_train = []
y_train_labels = []

for idx, row in train_df.iterrows():
    img_path = row['Filepath']
    label_name = row['Label']
    
    # Load grayscale image (tidak perlu resize karena file sudah 224x224 offline)
    img = Image.open(img_path)
    img_arr = np.array(img, dtype=np.float32)
    # Normalisasi ke [-1, 1]
    img_arr = (img_arr - 127.5) / 127.5
    
    X_train.append(img_arr)
    y_train_labels.append(class_to_idx[label_name])

X_train = np.expand_dims(np.array(X_train), axis=-1)
y_train_labels = np.array(y_train_labels, dtype=np.int32)

print("Dataset cGAN Train Shape:", X_train.shape)
print("Dataset cGAN Labels Shape:", y_train_labels.shape)


### 6. Melatih cGAN (Latihan GAN)


In [ ]:
GAN_EPOCHS = 10 # Setel default 10 agar cepat. Tingkatkan ke 50-100 untuk hasil maksimal.
BATCH_SIZE = 64

dataset = tf.data.Dataset.from_tensor_slices((X_train, y_train_labels)).shuffle(X_train.shape[0]).batch(BATCH_SIZE)

print(f"Memulai training cGAN selama {GAN_EPOCHS} Epoch...")
for epoch in range(GAN_EPOCHS):
    epoch_gen_loss = 0
    epoch_disc_loss = 0
    num_batches = 0
    
    for image_batch, label_batch in dataset:
        g_loss, d_loss = train_step(image_batch, label_batch)
        epoch_gen_loss += g_loss
        epoch_disc_loss += d_loss
        num_batches += 1
        
    print(f"Epoch {epoch+1}/{GAN_EPOCHS} - Gen Loss: {epoch_gen_loss/num_batches:.4f} - Disc Loss: {epoch_disc_loss/num_batches:.4f}")

print("Training cGAN selesai!")


### 7. Sintesis Gambar Baru (Data Balancing)


In [ ]:
# Tentukan folder penyimpanan gambar sintetis
synthetic_dir = './synthetic_images'
os.makedirs(synthetic_dir, exist_ok=True)

# Hitung target balancing berdasarkan kelas mayoritas (Normal)
train_counts = train_df['Label'].value_counts()
max_count = train_counts.max() # 2.671
print(f"Target jumlah data per kelas: {max_count} gambar")

synthetic_records = []

for label_name in class_names:
    current_count = train_counts[label_name]
    needed_count = max_count - current_count
    
    if needed_count > 0:
        print(f"Generating {needed_count} gambar sintetis untuk kelas {label_name}...")
        
        # Generator noise & label vectors
        noise = tf.random.normal([needed_count, LATENT_DIM])
        labels = tf.fill([needed_count, 1], class_to_idx[label_name])
        
        # Generate gambar
        gen_imgs = generator([noise, labels], training=False).numpy()
        # Denormalisasi [-1, 1] ke [0, 255]
        gen_imgs = (gen_imgs * 127.5 + 127.5).astype(np.uint8)
        
        # Simpan ke disk
        for i in range(needed_count):
            img_filename = f"syn_{label_name.lower()}_{i:05d}.jpg"
            img_filepath = os.path.join(synthetic_dir, img_filename)
            
            # Simpan menggunakan OpenCV
            cv2.imwrite(img_filepath, gen_imgs[i])
            
            synthetic_records.append({
                'Filepath': img_filepath,
                'Label': label_name
            })
            
print(f"Berhasil men-generate {len(synthetic_records)} gambar sintetis total!")

# Gabungkan data asli dan sintetis ke DataFrame training seimbang (Balanced)
syn_df = pd.DataFrame(synthetic_records)
train_balanced_df = pd.concat([train_df, syn_df], axis=0).reset_index(drop=True)

print("--- Distribusi Dataset Baru Setelah Balancing (Train Balanced) ---")
print(train_balanced_df['Label'].value_counts())


### 8. ImageDataGenerator & flow_from_dataframe (Balanced Data, Tanpa Bottleneck CPU)


In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32 # Dikembalikan ke 32 karena batch size 128 menyebabkan bottleneck baca disk pada Windows

# Preprocessing online kustom dihilangkan karena gambar sudah diproses CLAHE secara offline.
train_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input,
    rotation_range=20,
    zoom_range=0.15,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

test_datagen = tf.keras.preprocessing.image.ImageDataGenerator(
    preprocessing_function=tf.keras.applications.vgg16.preprocess_input
)

train_generator = train_datagen.flow_from_dataframe(
    dataframe=train_balanced_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=True,
    seed=42
)

val_test_generator = test_datagen.flow_from_dataframe(
    dataframe=val_test_df,
    x_col='Filepath',
    y_col='Label',
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)


### 9. Arsitektur Model LungNet22 (Fine-Tuned VGG16)


In [ ]:
# Load VGG16 Pretrained
vgg16_base = tf.keras.applications.VGG16(
    weights='imagenet', 
    include_top=False, 
    input_shape=(224, 224, 3)
)

vgg16_base.trainable = False

# Tambahkan Block 6 & 7 sesuai deskripsi paper
x = vgg16_base.output
x = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block6_conv1')(x)
x = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block6_conv2')(x)
x = tf.keras.layers.Conv2D(512, (3, 3), activation='relu', padding='same', name='block6_conv3')(x)
x = tf.keras.layers.GlobalAveragePooling2D(name='block6_pool')(x)

# Block 7: Flatten + Dense(6) + Softmax (dtype='float32' untuk mixed precision stability)
x = tf.keras.layers.Flatten(name='flatten')(x)
x = tf.keras.layers.Dense(6, name='dense_output')(x)
predictions = tf.keras.layers.Activation('softmax', dtype='float32', name='predictions')(x)

# Build Model
model = tf.keras.Model(inputs=vgg16_base.input, outputs=predictions)

# Compile model
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.000001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


### 10. Melatih Model Classifier (Training)


In [ ]:
EPOCHS = 100

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

checkpoint = tf.keras.callbacks.ModelCheckpoint(
    'lungnet22_skenario_b_best.h5',
    monitor='val_accuracy',
    save_best_only=True
)

history = model.fit(
    train_generator,
    validation_data=val_test_generator,
    epochs=EPOCHS,
    callbacks=[early_stopping, checkpoint]
)


### 11. Evaluasi Hasil: Kurva Akurasi & Loss


In [ ]:
# Plot Akurasi & Loss
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot Akurasi
axes[0].plot(history.history['accuracy'], label='Train Accuracy', color='blue', linewidth=2)
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy', color='orange', linewidth=2)
axes[0].set_title('Akurasi Model dari Epoch ke Epoch (Skenario B - Balanced)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Akurasi')
axes[0].legend()
axes[0].grid(True)

# Plot Loss
axes[1].plot(history.history['loss'], label='Train Loss', color='blue', linewidth=2)
axes[1].plot(history.history['val_loss'], label='Val Loss', color='orange', linewidth=2)
axes[1].set_title('Loss Model dari Epoch ke Epoch (Skenario B - Balanced)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()


### 12. Prediksi & Confusion Matrix


In [ ]:
# Prediksi
val_test_generator.reset()
Y_pred = model.predict(val_test_generator)
y_pred = np.argmax(Y_pred, axis=1)
y_true = val_test_generator.classes

# Plot Confusion Matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix — Skenario B (Balanced dengan GAN)')
plt.ylabel('Label Sebenarnya')
plt.xlabel('Label Prediksi')
plt.tight_layout()
plt.show()


### 13. Metrik Evaluasi Lengkap per Kelas (Presisi, Recall, Specificity, F1-Score)


In [ ]:
# Fungsi untuk menghitung Specificity per kelas
def calculate_specificity(cm):
    specificities = []
    num_classes = cm.shape[0]
    for i in range(num_classes):
        tp = cm[i, i]
        fn = np.sum(cm[i, :]) - tp
        fp = np.sum(cm[:, i]) - tp
        tn = np.sum(cm) - tp - fn - fp
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        specificities.append(specificity)
    return specificities

specs = calculate_specificity(cm)

# Hitung Precision, Recall, F1
report_dict = classification_report(y_true, y_pred, target_names=class_names, output_dict=True)

metrics_data = []
for i, name in enumerate(class_names):
    metrics_data.append({
        'Kelas': name,
        'Accuracy': report_dict[name].get('accuracy', report_dict['accuracy']),
        'Precision': report_dict[name]['precision'],
        'Recall (Sensitivity)': report_dict[name]['recall'],
        'Specificity': specs[i],
        'F1-Score': report_dict[name]['f1-score']
    })

metrics_df = pd.DataFrame(metrics_data)
print("--- Tabel Metrik Evaluasi per Kelas (Skenario B) ---")
display(metrics_df)

# Plot grafik batang metrik
metrics_df.set_index('Kelas')[['Precision', 'Recall (Sensitivity)', 'Specificity', 'F1-Score']].plot(kind='bar', figsize=(12, 6))
plt.title("Perbandingan Metrik Evaluasi per Kelas — Skenario B")
plt.ylabel("Nilai Metrik (0.0 - 1.0)")
plt.xlabel("Kelas")
plt.ylim(0, 1.05)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


### 14. Kurva ROC & Nilai AUC per Kelas


In [ ]:
# Binarize label true untuk perhitungan ROC multiclass (One-Vs-Rest)
y_true_bin = label_binarize(y_true, classes=np.arange(len(class_names)))

plt.figure(figsize=(10, 8))
for i in range(len(class_names)):
    fpr, tpr, _ = roc_curve(y_true_bin[:, i], Y_pred[:, i])
    roc_auc = auc(fpr, tpr)
    plt.plot(fpr, tpr, label=f'ROC {class_names[i]} (AUC = {roc_auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Tebakan Acak (AUC = 0.50)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)')
plt.ylabel('True Positive Rate (Sensitivity)')
plt.title('Receiver Operating Characteristic (ROC) Curve per Kelas (Skenario B)')
plt.legend(loc="lower right")
plt.grid(True)
plt.show()


### 15. Visualisasi Interpretability: Grad-CAM & Score-CAM


In [ ]:
# Implementasi Grad-CAM
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = tf.keras.models.Model(
        [model.inputs], 
        [model.get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Implementasi Score-CAM (Gradient-Free)
def make_scorecam_heatmap(img_array, model, last_conv_layer_name, pred_index=None, max_channels=64):
    feature_extractor = tf.keras.models.Model(
        [model.inputs],
        [model.get_layer(last_conv_layer_name).output]
    )
    feature_maps = feature_extractor(img_array)[0]
    
    if pred_index is None:
        preds = model(img_array)
        pred_index = tf.argmax(preds[0])
        
    h_feat, w_feat, c_feat = feature_maps.shape
    
    # Batasi channel untuk efisiensi
    if c_feat > max_channels:
        stds = np.std(feature_maps, axis=(0, 1))
        top_indices = np.argsort(stds)[::-1][:max_channels]
        selected_features = tf.gather(feature_maps, top_indices, axis=-1)
    else:
        selected_features = feature_maps
        top_indices = np.arange(c_feat)
        
    num_channels = selected_features.shape[-1]
    input_size = img_array.shape[1:3]
    
    upsampled_features = tf.image.resize(selected_features, input_size, method='bilinear')
    
    min_val = tf.reduce_min(upsampled_features, axis=(0, 1), keepdims=True)
    max_val = tf.reduce_max(upsampled_features, axis=(0, 1), keepdims=True)
    norm_features = (upsampled_features - min_val) / (max_val - min_val + 1e-10)
    
    norm_features_expanded = tf.transpose(norm_features, perm=[2, 0, 1])
    norm_features_expanded = tf.expand_dims(norm_features_expanded, axis=-1)
    
    input_repeated = tf.repeat(img_array, num_channels, axis=0)
    masked_images = input_repeated * norm_features_expanded
    
    scores = []
    for start_idx in range(0, num_channels, 32):
        end_idx = min(start_idx + 32, num_channels)
        batch_masked = masked_images[start_idx:end_idx]
        batch_preds = model(batch_masked)
        batch_scores = batch_preds[:, pred_index]
        scores.extend(batch_scores.numpy())
        
    scores = np.array(scores)
    
    weighted_features = selected_features.numpy() * scores[np.newaxis, np.newaxis, :]
    heatmap = np.sum(weighted_features, axis=-1)
    
    heatmap = np.maximum(heatmap, 0)
    heatmap = heatmap / (np.max(heatmap) + 1e-10)
    
    return heatmap

# Fungsi untuk menumpangkan heatmap ke citra asli
def overlay_heatmap(img_path, heatmap, alpha=0.4, colormap=cv2.COLORMAP_JET):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (224, 224))
    heatmap_resized = cv2.resize(heatmap, (224, 224))
    heatmap_colored = np.uint8(255 * heatmap_resized)
    heatmap_colored = cv2.applyColorMap(heatmap_colored, colormap)
    superimposed_img = cv2.addWeighted(img, 1 - alpha, heatmap_colored, alpha, 0)
    return cv2.cvtColor(superimposed_img, cv2.COLOR_BGR2RGB)


### 16. Menampilkan Hasil Grad-CAM & Score-CAM Berdampingan (Skenario B)


In [ ]:
# Ambil 3 sampel acak
sample_rows = val_test_df.sample(n=3, random_state=42).reset_index(drop=True)

fig, axes = plt.subplots(3, 3, figsize=(15, 15))
last_conv_layer_name = 'block6_conv3'

for i, row in sample_rows.iterrows():
    img_path = row['Filepath']
    true_label = row['Label']
    
    # Load & preprocess
    img_pil = Image.open(img_path).resize((224, 224))
    img_arr = np.array(img_pil)
    if len(img_arr.shape) == 2:
        img_arr = np.stack([img_arr, img_arr, img_arr], axis=-1)
        
    # Di preprocessed folder, CLAHE sudah dilakukan secara offline.
    img_preprocessed = tf.keras.applications.vgg16.preprocess_input(img_arr.astype(np.float32))
    img_input = np.expand_dims(img_preprocessed, axis=0)
    
    # Predict
    preds = model.predict(img_input)
    pred_idx = np.argmax(preds[0])
    pred_label = class_names[pred_idx]
    
    # Heatmaps
    gradcam_heat = make_gradcam_heatmap(img_input, model, last_conv_layer_name, pred_index=pred_idx)
    scorecam_heat = make_scorecam_heatmap(img_input, model, last_conv_layer_name, pred_index=pred_idx, max_channels=64)
    
    # Overlays
    gradcam_img = overlay_heatmap(img_path, gradcam_heat)
    scorecam_img = overlay_heatmap(img_path, scorecam_heat)
    
    img_orig = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
    img_orig = cv2.resize(img_orig, (224, 224))
    
    # Plot original
    axes[i, 0].imshow(img_orig)
    axes[i, 0].set_title(f"Original Image\nTrue: {true_label}")
    axes[i, 0].axis('off')
    
    # Plot Grad-CAM
    axes[i, 1].imshow(gradcam_img)
    axes[i, 1].set_title(f"Grad-CAM\nPred: {pred_label} ({preds[0][pred_idx]*100:.1f}%)")
    axes[i, 1].axis('off')
    
    # Plot Score-CAM
    axes[i, 2].imshow(scorecam_img)
    axes[i, 2].set_title(f"Score-CAM\nPred: {pred_label} ({preds[0][pred_idx]*100:.1f}%)")
    axes[i, 2].axis('off')

plt.tight_layout()
plt.show()
